# Thermal Drift & Mode Density Analysis

Explore how thermal fluctuations limit the number of distinguishable
resonant modes, and how ZIM integration recovers storage density.

From WCFOMA Addendum (February 2026).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from simulations.thermal import (
    run_standard_thermal_comparison,
    sensitivity_sweep,
    analyze_thermal_drift,
)
from simulations.common import ThermalParams, MicroCellParams
from analysis.plotting import apply_style, COLORS

apply_style()
print('Thermal Analysis — Ready')

## Baseline: Normal vs ZIM

In [ ]:
results = run_standard_thermal_comparison()

for label, r in results.items():
    print(f'{label}: {r.max_safe_modes} modes, {r.density_tb_per_cm3:.2f} Tb/cm³')

r_n = results['Without ZIM']
r_z = results['With ZIM']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(r_n.mode_numbers, r_n.margins / 1e6, color=COLORS['normal'],
         label=f'Normal ({r_n.max_safe_modes} modes)')
ax1.plot(r_z.mode_numbers, r_z.margins / 1e6, color=COLORS['zim'],
         label=f'ZIM ({r_z.max_safe_modes} modes)')
ax1.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax1.set_xlabel('Mode Number')
ax1.set_ylabel('Margin (MHz)')
ax1.set_title('Mode Distinguishability Margin')
ax1.legend()
ax1.set_xlim(0, r_z.max_safe_modes * 1.2)

ax2.plot(r_n.mode_numbers, r_n.drift_ranges / 1e6, color=COLORS['normal_stressed'],
         label='Thermal drift (Normal)', alpha=0.7)
ax2.plot(r_z.mode_numbers, r_z.drift_ranges / 1e6, color=COLORS['zim_stressed'],
         label='Thermal drift (ZIM)', alpha=0.7)
ax2.plot(r_n.mode_numbers, r_n.linewidths / 1e6, '--', color=COLORS['theory'],
         label='Linewidth')
ax2.set_xlabel('Mode Number')
ax2.set_ylabel('Width (MHz)')
ax2.set_title('Drift vs Linewidth')
ax2.legend()
ax2.set_xlim(0, r_z.max_safe_modes * 1.2)

plt.tight_layout()
plt.show()

## Sensitivity Sweeps

How do key parameters affect the storage density ceiling?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Q sweep
q_vals, q_modes, q_dens = sensitivity_sweep('Q', np.linspace(100, 5000, 50), use_zim=True)
axes[0, 0].plot(q_vals, q_dens, color=COLORS['zim'])
axes[0, 0].set_xlabel('Quality Factor Q')
axes[0, 0].set_ylabel('Density (Tb/cm³)')
axes[0, 0].set_title('Storage Density vs Q (with ZIM)')
axes[0, 0].axhline(y=1, color='red', linestyle='--', alpha=0.3, label='1 Tb/cm³')
axes[0, 0].legend()

# ΔT sweep
dt_vals, dt_modes, dt_dens = sensitivity_sweep('delta_T', np.linspace(1, 20, 40), use_zim=True)
axes[0, 1].plot(dt_vals, dt_dens, color=COLORS['zim_stressed'])
axes[0, 1].set_xlabel('Temperature Variation ΔT (K)')
axes[0, 1].set_ylabel('Density (Tb/cm³)')
axes[0, 1].set_title('Storage Density vs ΔT (with ZIM)')

# α sweep
a_vals, a_modes, a_dens = sensitivity_sweep('alpha', np.linspace(0.001, 0.005, 40), use_zim=True)
axes[1, 0].plot(a_vals * 100, a_dens, color=COLORS['normal_stressed'])
axes[1, 0].set_xlabel('Thermal Drift Coefficient α (%/K)')
axes[1, 0].set_ylabel('Density (Tb/cm³)')
axes[1, 0].set_title('Storage Density vs α (with ZIM)')

# ZIM reduction sweep
r_vals, r_modes, r_dens = sensitivity_sweep('drift_reduction_zim', np.linspace(5, 50, 40), use_zim=True)
axes[1, 1].plot(r_vals, r_dens, color=COLORS['zim'])
axes[1, 1].set_xlabel('ZIM Drift Reduction Factor')
axes[1, 1].set_ylabel('Density (Tb/cm³)')
axes[1, 1].set_title('Storage Density vs ZIM Effectiveness')

plt.tight_layout()
plt.show()

print('\nKey insight: Q and ZIM reduction factor are the strongest levers.')
print(f'At Q=2000 with ZIM: ~{q_dens[np.argmin(np.abs(q_vals-2000))]:.1f} Tb/cm³')
print(f'At ΔT=10K with ZIM: ~{dt_dens[np.argmin(np.abs(dt_vals-10))]:.1f} Tb/cm³')